<a href="https://colab.research.google.com/github/mariolopezguasp/SP500Prediction/blob/main/2ModeloML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler


df = pd.read_csv('dataset_ia_log_returns_10y.csv', index_col=0, parse_dates=True)

X = df.iloc[:-1].values
y = df.iloc[1:].values

n = len(X)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

# Apply StandardScaler to the features
scaler =StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train size: {X_train.shape}, Val size: {X_val.shape}, Test size: {X_test.shape}")

Train size: (1757, 12), Val size: (376, 12), Test size: (377, 12)


In [4]:

import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
import json

# Train XGBoost Model
# Since we have multiple outputs, we use MultiOutputRegressor
xgb_estimator = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=50, max_depth=3, learning_rate=0.1)
model = MultiOutputRegressor(xgb_estimator)
model.fit(X_train, y_train)

# Calculate number of parameters (approximate via number of trees * nodes)
num_trees = sum(estimator.get_num_boosting_rounds() for estimator in model.estimators_)
print(f"Total number of trees built: {num_trees}")

# Evaluate
y_train_pred = model.predict(X_train)
y_val_pred = model.predict(X_val)
y_test_pred = model.predict(X_test)

mse_train = mean_squared_error(y_train, y_train_pred)
mse_val = mean_squared_error(y_val, y_val_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f"Train MSE: {mse_train:.6f}")
print(f"Validation MSE: {mse_val:.6f}")
print(f"Test MSE: {mse_test:.6f}")

results_xgb = {'train_mse': mse_train, 'val_mse': mse_val, 'test_mse': mse_test, 'params': f"{num_trees} trees"}
with open('results_xgb.json', 'w') as f: json.dump(results_xgb, f)

# 1. Calcular e interpretar el RMSE (Raíz del Error Cuadrático Medio)
rmse_train = np.sqrt(mse_train)
rmse_val = np.sqrt(mse_val)
rmse_test = np.sqrt(mse_test)

print("--- Root Mean Squared Error (RMSE) ---")
print(f"Train RMSE: {rmse_train:.6f} (Desviación media del modelo: {rmse_train*100:.2f}%)")
print(f"Validation RMSE: {rmse_val:.6f} (Desviación media del modelo: {rmse_val*100:.2f}%)")
print(f"Test RMSE: {rmse_test:.6f} (Desviación media del modelo: {rmse_test*100:.2f}%)")
print("-" * 38)


Total number of trees built: 600
Train MSE: 0.000262
Validation MSE: 0.000273
Test MSE: 0.000332
--- Root Mean Squared Error (RMSE) ---
Train RMSE: 0.016187 (Desviación media del modelo: 1.62%)
Validation RMSE: 0.016519 (Desviación media del modelo: 1.65%)
Test RMSE: 0.018227 (Desviación media del modelo: 1.82%)
--------------------------------------


In [5]:
# 1. Calcular e interpretar el RMSE (Raíz del Error Cuadrático Medio)
rmse_train = np.sqrt(mse_train)
rmse_val = np.sqrt(mse_val)
rmse_test = np.sqrt(mse_test)

print("--- Root Mean Squared Error (RMSE) ---")
print(f"Train RMSE: {rmse_train:.6f} (Desviación media del modelo: {rmse_train*100:.2f}%)")
print(f"Validation RMSE: {rmse_val:.6f} (Desviación media del modelo: {rmse_val*100:.2f}%)")
print(f"Test RMSE: {rmse_test:.6f} (Desviación media del modelo: {rmse_test*100:.2f}%)")
print("-" * 38)

--- Root Mean Squared Error (RMSE) ---
Train RMSE: 0.016187 (Desviación media del modelo: 1.62%)
Validation RMSE: 0.016519 (Desviación media del modelo: 1.65%)
Test RMSE: 0.018227 (Desviación media del modelo: 1.82%)
--------------------------------------


In [6]:
import numpy as np

# Calculate directional accuracy for training set
signo_real_train = np.sign(y_train)
signo_pred_train = np.sign(y_train_pred)
accuracy_direccional_train = np.mean(signo_real_train == signo_pred_train)

# Calculate directional accuracy for validation set
signo_real_val = np.sign(y_val)
signo_pred_val = np.sign(y_val_pred)
accuracy_direccional_val = np.mean(signo_real_val == signo_pred_val)

# Calculate directional accuracy for test set
signo_real_test = np.sign(y_test)
signo_pred_test = np.sign(y_test_pred)
accuracy_direccional_test = np.mean(signo_real_test == signo_pred_test)

print("--- Accuracy Direccional ---")
print(f"Train Directional Accuracy: {accuracy_direccional_train*100:.2f}%")
print(f"Validation Directional Accuracy: {accuracy_direccional_val*100:.2f}%")
print(f"Test Directional Accuracy: {accuracy_direccional_test*100:.2f}%")
print("----------------------------")

--- Accuracy Direccional ---
Train Directional Accuracy: 60.68%
Validation Directional Accuracy: 51.55%
Test Directional Accuracy: 51.02%
----------------------------


In [9]:
import numpy as np

# --- Directional Accuracy for Training Set ---
signo_real_train = np.sign(y_train)
signo_pred_train = np.sign(y_train_pred)
accuracy_direccional_train = np.mean(signo_real_train == signo_pred_train)

# --- Directional Accuracy for Validation Set ---
signo_real_val = np.sign(y_val)
signo_pred_val = np.sign(y_val_pred)
accuracy_direccional_val = np.mean(signo_real_val == signo_pred_val)

# --- Directional Accuracy for Test Set ---
signo_real_test = np.sign(y_test)
signo_pred_test = np.sign(y_test_pred)
accuracy_direccional_test = np.mean(signo_real_test == signo_pred_test)

print("--- Accuracy Direccional ---")
print(f"Train Directional Accuracy: {accuracy_direccional_train*100:.2f}%")
print(f"Validation Directional Accuracy: {accuracy_direccional_val*100:.2f}%")
print(f"Test Directional Accuracy: {accuracy_direccional_test*100:.2f}%")

--- Accuracy Direccional ---
Train Directional Accuracy: 60.68%
Validation Directional Accuracy: 51.55%
Test Directional Accuracy: 51.02%


In [6]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

# Assuming 'model' is already trained from a previous cell.
# If not, you would need to re-run the training cell.

if 'model' in locals():
    num_trees = sum(estimator.get_num_boosting_rounds() for estimator in model.estimators_)
    print(f"Total number of trees built (approximate parameters): {num_trees}")
else:
    print("Error: The 'model' variable is not defined. Please ensure the model training cell has been run.")

Total number of trees built (approximate parameters): 600
